# 03 · Aggregation and extremizing

Asking the model several times and combining the answers beats a single answer — but naive averaging is systematically timid, which is what extremizing corrects for.

In [ ]:
import sys

sys.path.insert(0, "../src")
import numpy as np

from forecaster.aggregate import extremize, geometric_mean_of_odds, mean, median, trimmed_mean
from forecaster.scoring import brier_score

## Combination methods on noisy samples

Simulate 5 noisy samples around a true probability and compare combination methods.

In [ ]:
rng = np.random.default_rng(0)
true_p = 0.8


def sample_noisy(true_p, noise=0.1, k=5):
    logit = np.log(true_p / (1 - true_p))
    noisy_logits = logit + rng.normal(0, noise, k)
    return 1 / (1 + np.exp(-noisy_logits))


samples = sample_noisy(true_p, noise=0.3)
print("samples:", np.round(samples, 2))
print("mean:", mean(samples))
print("median:", median(samples))
print("trimmed mean:", trimmed_mean(samples, 0.2))
print("geometric mean of odds:", geometric_mean_of_odds(samples))

## Extremizing

Averaging several independent noisy estimates pulls the combined probability toward 0.5 more than it should — each sample has seen only partial evidence, so naive combination is systematically timid. Extremizing multiplies the combined log-odds by `alpha > 1` to push back out.

In [ ]:
combined = geometric_mean_of_odds(samples)
for alpha in [1.0, 1.5, 2.0, 3.0]:
    print(f"alpha={alpha}: {extremize(combined, alpha):.3f}")

## Fitting alpha on earlier questions, testing on later ones

In [ ]:
def simulate_question_set(n, noise, seed):
    rng = np.random.default_rng(seed)
    true_ps = rng.uniform(0.1, 0.9, n)
    outcomes = (rng.random(n) < true_ps).astype(float)
    combined = np.array([geometric_mean_of_odds(sample_noisy(tp, noise=noise)) for tp in true_ps])
    return combined, outcomes


train_combined, train_outcomes = simulate_question_set(200, noise=0.3, seed=1)
test_combined, test_outcomes = simulate_question_set(200, noise=0.3, seed=2)

best_alpha, best_brier = 1.0, brier_score(train_combined, train_outcomes)
for alpha in np.linspace(1.0, 3.0, 21):
    extremized = np.array([extremize(p, alpha) for p in train_combined])
    b = brier_score(extremized, train_outcomes)
    if b < best_brier:
        best_alpha, best_brier = alpha, b
print("best alpha on train:", round(best_alpha, 2))

test_extremized = np.array([extremize(p, best_alpha) for p in test_combined])
print("test Brier, no extremizing:", brier_score(test_combined, test_outcomes))
print("test Brier, with fitted alpha:", brier_score(test_extremized, test_outcomes))

**Try it:** set `noise=0.2` in `simulate_question_set` above and see whether extremizing still helps as much.